# ProboSed — 02: Stochastic Slope Stability Model

**IODP Expedition 405 — Site C0019J, Japan Trench**

Runs the coupled Ornstein-Uhlenbeck slope stability model and produces three figures:
- Slope-only vs fault-coupled trajectory ensembles and failure probability
- Lyapunov exponent estimate with warmup exclusion
- Sensitivity analysis across physically motivated parameter ranges

**Governing equations:**

    ds = -k_f * s * dt + sigma_s * dW_s
    dq = (-gamma * q + alpha * s) * dt + sigma_q * dW_q
    Failure when: q(t) > theta(x),  theta = score * 2/3

**Physical parameter context (IODP Expeditions 343, 405, 386):**
- Frontal prism Vp = 1550-1750 m/s, resistivity = 0.5-1.8 Ohm-m (Exp 405 LWD Unit I)
- Frontal prism Pc' = 17 MPa, significant overconsolidation (Exp 343, Valdez et al. 2015)
- Coseismic slip ~50 m at C0019 decollement, 2011 Mw9 Tohoku-oki (Fulton et al. 2013)
- Seismic strengthening of background hemipelagic sediment (Exp 386)

In [ ]:
# Cell 1: Install dependencies
!pip install numpy matplotlib pandas --quiet

In [ ]:
# Cell 2: Mount Google Drive and clone ProboSed
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
repo_url  = 'https://github.com/yourusername/ProboSed.git'  # update this URL
repo_path = '/content/ProboSed'
result = subprocess.run(['git', 'clone', repo_url, repo_path],
                        capture_output=True, text=True)
if result.returncode != 0:
    subprocess.run(['git', '-C', repo_path, 'pull'], capture_output=True)
sys.path.insert(0, repo_path)
print('ProboSed modules available')

In [ ]:
# Cell 3: Configure output directory
import os
OUTPUT_DIR = '/content/drive/MyDrive/iodp/X405/ProboSed_Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

## Simulation Parameters

All physical parameters are defined at the top of `stability.py`.
The values below match the defaults. Edit `BASE` to explore alternatives.

| Parameter | Value | Physical context |
|---|---|---|
| gamma | 1.0 | Slope damping — moderate restoring force |
| sigma_q | 0.6 | Slope noise — weak prism material (Vp 1550-1750 m/s) |
| alpha | 0.5 | Fault-slope coupling coefficient |
| slip_mag | 3.0 | Mainshock impulse (scaled from ~50 m coseismic slip) |
| threshold | 1.0 | Failure threshold — between VCD scores 1 and 2 |

In [ ]:
# Cell 4: Import modules and define base parameters
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from slope.stability import (run_ensemble, calculate_lyapunov,
                       failure_statistics, threshold_from_vcd, run_sensitivity)

# base parameters — match toy_model.py defaults
BASE = dict(
    T=10.0, dt=0.01,
    gamma=1.0, sigma_q=0.6, sigma_s=0.5,
    k_f=1.0, alpha=0.5, threshold=1.0,
    slip_mag=3.0, mainshock_t=5.0, seed=42
)
print('Modules imported, parameters defined')

In [ ]:
# Cell 5: Run slope-only ensemble (no fault coupling, background noise only)
# This is the null model — slope displacement under tectonic noise without
# a mainshock event. alpha=0 removes the fault coupling term.
print('Running slope-only ensemble (N=5000)...')
q_solo, s_solo, p_fail_solo, trans_solo = run_ensemble(
    N_paths=5000, alpha=0.0, **{k: v for k, v in BASE.items()
                                 if k not in ['alpha']}
)
print(f'Slope-only P(failure): {p_fail_solo:.3f}')

In [ ]:
# Cell 6: Run fault-coupled ensemble (Tohoku-style mainshock impulse)
# alpha=0.5 couples fault slip to slope displacement.
# slip_mag=3.0 is the dimensionless scaled equivalent of ~50 m coseismic slip.
print('Running fault-coupled ensemble (N=5000)...')
q_coupled, s_coupled, p_fail_coupled, trans_coupled = run_ensemble(
    N_paths=5000, **BASE
)
print(f'Fault-coupled P(failure): {p_fail_coupled:.3f}')
print(f'Failure probability increase: {(p_fail_coupled - p_fail_solo):.3f} '
      f'({p_fail_coupled/p_fail_solo:.1f}x multiplier)')

In [ ]:
# Cell 7: Estimate Lyapunov exponent
# The warmup fraction excludes the initial transient where all trajectories
# start at q=0, causing spurious log-growth in the first few timesteps.
lyapunov, log_growth = calculate_lyapunov(q_coupled, BASE['dt'], warmup_fraction=0.01)
print(f'Lyapunov exponent (post-warmup): {lyapunov:.4f} per unit time')
if lyapunov > 0:
    print('  Positive: system exhibits sensitivity to initial conditions')
else:
    print('  Non-positive: system is stable / convergent')

In [ ]:
# Cell 8: Failure statistics
stats = failure_statistics(q_coupled, trans_coupled, BASE['threshold'], BASE['dt'])
print('\n--- Failure Statistics (fault-coupled ensemble) ---')
for k, v in stats.items():
    if isinstance(v, float):
        print(f'  {k:<25} {v:.4f}')
    else:
        print(f'  {k:<25} {v}')

## Figure 1 — Trajectory Ensembles and Transport Distribution

In [ ]:
# Cell 9: Figure 1 — three-panel summary
time = np.linspace(0, BASE['T'], int(BASE['T']/BASE['dt']) + 1)
colors = plt.cm.viridis(np.linspace(0, 1, 20))

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.patch.set_facecolor('white')
for ax in axes:
    ax.set_facecolor('white')

# Panel A: slope-only
for i, c in enumerate(colors):
    axes[0].plot(time, q_solo[:, i], color=c, alpha=0.75, lw=1.2)
axes[0].axhline(BASE['threshold'], color='red', ls='--', lw=2, label='Failure threshold')
axes[0].axvline(BASE['mainshock_t'], color='orange', ls=':', lw=2, label='Mainshock time')
axes[0].set_xlabel('Time (simulation units)', fontsize=13)
axes[0].set_ylabel('Slope displacement q(t)', fontsize=13)
axes[0].set_title(f'(A) Slope-Only Trajectories\nP(failure) = {p_fail_solo:.2f}',
                  fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Panel B: fault-coupled
for i, c in enumerate(colors):
    axes[1].plot(time, q_coupled[:, i], color=c, alpha=0.75, lw=1.2)
axes[1].axhline(BASE['threshold'], color='red', ls='--', lw=2, label='Failure threshold')
axes[1].axvline(BASE['mainshock_t'], color='orange', ls=':', lw=2, label='Mainshock impulse')
axes[1].set_xlabel('Time (simulation units)', fontsize=13)
axes[1].set_ylabel('Slope displacement q(t)', fontsize=13)
axes[1].set_title(f'(B) Fault-Coupled Trajectories\nP(failure) = {p_fail_coupled:.2f}',
                  fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

# Panel C: transported sediment distribution
axes[2].hist(trans_coupled, bins=25, edgecolor='k', color='navy', alpha=0.7)
axes[2].axvline(trans_coupled.mean(), color='red', ls='--', lw=2,
                label=f'Mean = {trans_coupled.mean():.3f}')
axes[2].set_xlabel('Transported sediment (dimensionless model units)', fontsize=13)
axes[2].set_ylabel('Count', fontsize=13)
axes[2].set_title('(C) Transported Sediment Distribution\n(Time-integrated threshold exceedance)',
                  fontsize=14, fontweight='bold')
axes[2].legend(fontsize=11)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
out = os.path.join(OUTPUT_DIR, 'toy_model_summary.png')
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out}')

## Figure 2 — Failure Probability Comparison

In [ ]:
# Cell 10: Figure 2 — failure probability bar chart
fig2, ax2 = plt.subplots(figsize=(7, 6))
fig2.patch.set_facecolor('white')
ax2.set_facecolor('white')

bars = ax2.bar(
    ['Slope Only\n(no fault coupling)', 'Slope + Fault\n(Tohoku impulse)'],
    [p_fail_solo, p_fail_coupled],
    color=['steelblue', 'tomato'], edgecolor='k', alpha=0.85, width=0.5
)
for bar, val in zip(bars, [p_fail_solo, p_fail_coupled]):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02,
             f'{val:.2f}', ha='center', fontsize=14, fontweight='bold')
ax2.set_ylabel('Failure probability', fontsize=13)
ax2.set_ylim(0, 1.15)
ax2.set_title('Effect of Fault Slip Coupling on Slope Failure\nSite C0019J, Japan Trench',
              fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
out = os.path.join(OUTPUT_DIR, 'toy_model_failure_probability.png')
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out}')

## Figure 3 — Lyapunov Exponent

The shaded gray region marks the warmup period excluded from the mean.
This exclusion is critical: all trajectories start at q=0, creating a
spurious log-growth spike in the first few timesteps that inflates the
Lyapunov estimate if included.

In [ ]:
# Cell 11: Figure 3 — Lyapunov exponent over time
time_lyap  = np.linspace(0, BASE['T'], len(log_growth))
warmup_end = int(0.01 * len(log_growth))

fig3, ax3 = plt.subplots(figsize=(10, 5))
fig3.patch.set_facecolor('white')
ax3.set_facecolor('white')

ax3.axvspan(0, time_lyap[warmup_end], color='lightgray', alpha=0.5,
            label='Warmup (excluded from LE mean)')
ax3.plot(time_lyap, log_growth, color='#2c3e50', lw=1.5, alpha=0.8,
         label='Log-growth rate of perturbations')
ax3.axhline(0, color='red', ls='--', lw=1.5, alpha=0.6,
            label='Zero (stability boundary)')
ax3.axhline(lyapunov, color='navy', ls='-', lw=2,
            label=f'LE mean (post-warmup) = {lyapunov:.4f} / time unit')
ax3.axvline(BASE['mainshock_t'], color='orange', ls=':', lw=2, label='Mainshock')
ax3.set_xlabel('Time (simulation units)', fontsize=13)
ax3.set_ylabel('Mean log-growth rate of perturbations', fontsize=13)
ax3.set_title('Lyapunov Exponent Estimate\nFault-Coupled Ensemble, Site C0019J, Japan Trench',
              fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

plt.tight_layout()
out = os.path.join(OUTPUT_DIR, 'toy_model_lyapunov.png')
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out}')

## VCD Stability Index — Threshold Mapping

The stability index maps VCD disturbance scores to failure thresholds.
This cell shows how the threshold changes across the four VCD categories
and computes failure probability for each, demonstrating that the model
responds as expected to the ordinal score.

In [ ]:
# Cell 12: Show how failure probability varies with VCD score
print('VCD Score | Disturbance      | Threshold | P(failure)')
print('-' * 55)
labels = ['Slurried/MTD', 'Scaly fabric', 'Coherent block', 'Intact bedding']
for score in [0, 1, 2, 3]:
    theta = threshold_from_vcd(score)
    _, _, p, _ = run_ensemble(N_paths=2000, threshold=float(theta), **
                              {k: v for k, v in BASE.items() if k != 'threshold'})
    print(f'{score:9d} | {labels[score]:<16} | {float(theta):9.2f} | {p:.3f}')